# Descarga y preparación de datos ERA5-Land

Este notebook documenta el proceso completo de descarga y preparación de los datos brutos.  
Solo es necesario ejecutarlo una vez. El resultado es el fichero `datos/temperaturas_verano_completo.csv`  
que utiliza el notebook de análisis `02_noches_tropicales_analisis.ipynb`.

**Fuente:** ERA5-Land — Copernicus Climate Data Store (CDS)  
**Variable:** Temperatura a 2m a las 06:00h (proxy de temperatura mínima nocturna)  
**Periodo:** 1980–2024, meses de verano (junio–septiembre)  
**Área:** Galicia (bbox: N44.0 / W-9.5 / S41.8 / E-6.5)

### Requisito previo

Necesitas una cuenta gratuita en el Copernicus Climate Data Store:  
https://cds.climate.copernicus.eu

Una vez registrado, obtén tu token personal en:  
https://cds.climate.copernicus.eu/profile

Pégalo en la celda de configuración más abajo.

## 0. Configuración de entorno

In [ ]:
import os
import sys

# Detectar si estamos en Google Colab o en local
EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/Colab_Notebooks/portfolio_clima/'
else:
    RUTA_BASE = '../'

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
print(f"Ruta base: {RUTA_BASE}")

In [ ]:
# Instalar el cliente de CDS si no está disponible
!pip install cdsapi netCDF4 cftime --quiet

import cdsapi
import zipfile
import xarray as xr
import pandas as pd
import numpy as np
import os
import time

print("Librerías cargadas correctamente")

In [ ]:
# ============================================================
# CONFIGURACIÓN — pega aquí tu token personal de CDS
# Encuéntralo en: https://cds.climate.copernicus.eu/profile
# ============================================================
TOKEN = "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"

# Crear el fichero de configuración que necesita cdsapi
config = f"url: https://cds.climate.copernicus.eu/api\nkey: {TOKEN}"
with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write(config)

print("Configuración de CDS guardada")

## 1. Descarga de datos ERA5-Land

Se descarga un fichero ZIP por año con la temperatura a las 06:00h  
durante los meses de verano (junio–septiembre) para el bounding box de Galicia.

**Tiempo estimado:** 5–20 minutos por año dependiendo de la cola de CDS.  
Para 44 años el proceso puede tardar varias horas.  

El bucle detecta los años ya descargados y los salta automáticamente,  
por lo que se puede interrumpir y reanudar sin perder trabajo.

### Mantener Colab activo durante la descarga

Para evitar que Colab se desconecte por inactividad, abre la consola del navegador  
con F12 → pestaña Console y ejecuta:

```javascript
setInterval(() => {document.querySelector("colab-connect-button").click()}, 60000)
```

In [ ]:
c = cdsapi.Client()

# Años a descargar
años = list(range(1980, 2025))

completados = []
fallidos = []

for año in años:
    ruta_fichero = RUTA_BASE + f'era5_diario_{año}.zip'

    # Si ya está descargado lo saltamos
    if os.path.exists(ruta_fichero):
        print(f"{año} — ya existe, saltando")
        completados.append(año)
        continue

    print(f"{año} — descargando...")

    try:
        c.retrieve(
            'reanalysis-era5-land',
            {
                'variable': '2m_temperature',
                'year': str(año),
                'month': ['06', '07', '08', '09'],
                'day': [f'{d:02d}' for d in range(1, 32)],
                'time': '06:00',
                'area': [44.0, -9.5, 41.8, -6.5],
                'data_format': 'netcdf',
                'download_format': 'zip',
            },
            ruta_fichero
        )
        print(f"{año} — completado")
        completados.append(año)
        time.sleep(5)

    except Exception as e:
        print(f"{año} — ERROR: {e}")
        fallidos.append(año)
        time.sleep(10)

print(f"\nResumen:")
print(f"  Completados: {len(completados)} años")
print(f"  Fallidos: {len(fallidos)} años")
if fallidos:
    print(f"  Años con error: {fallidos}")

## 2. Nota técnica — Problema con A Coruña

ERA5-Land es como un tablero de ajedrez puesto encima de Galicia. Cada casilla mide aproximadamente 9×9 km y solo las casillas que caen mayoritariamente sobre tierra tienen datos. Las que caen sobre el mar tienen NaN.

A Coruña está en una península muy estrecha. La casilla del tablero que le corresponde por coordenadas queda mayoritariamente sobre el mar, así que ERA5 la deja vacía. La celda siguiente busca el punto terrestre con datos válidos más cercano.

In [ ]:
# Verificar el problema de A Coruña usando el primer ZIP disponible
zips_disponibles = sorted([
    f for f in os.listdir(RUTA_BASE)
    if f.startswith('era5_diario_') and f.endswith('.zip')
])

ruta_temp = '/content/temp_check/'
os.makedirs(ruta_temp, exist_ok=True)

# Abrir el primer ZIP para inspeccionar la cuadrícula
with zipfile.ZipFile(RUTA_BASE + zips_disponibles[0], 'r') as z:
    nombre_nc = z.namelist()[0]
    z.extractall(ruta_temp)

ds_test = xr.open_dataset(ruta_temp + nombre_nc)

# Extraer la cuadrícula del primer día
t2m_dia1 = ds_test['t2m'].isel(valid_time=0)
df_grid = t2m_dia1.to_dataframe().reset_index().dropna(subset=['t2m'])

# Buscar los 10 puntos con datos más cercanos a A Coruña
lat_obj, lon_obj = 43.37, -8.40
zona = df_grid[
    (df_grid['latitude'] >= 43.0) & (df_grid['latitude'] <= 43.8) &
    (df_grid['longitude'] >= -9.0) & (df_grid['longitude'] <= -7.8)
].copy()
zona['distancia'] = ((zona['latitude'] - lat_obj)**2 + (zona['longitude'] - lon_obj)**2)**0.5

print("10 puntos con datos válidos más cercanos a A Coruña:")
print(zona.sort_values('distancia')[['latitude', 'longitude', 't2m', 'distancia']].head(10).round(4))

ds_test.close()
os.remove(ruta_temp + nombre_nc)

El punto con datos válidos más cercano a A Coruña es **lat=43.3, lon=-8.4**,  
que es el que se usa en el proceso de extracción.

## 3. Procesado — Unir todos los ZIPs en un único CSV

In [ ]:
# Coordenadas finales de las 5 ciudades
# A Coruña usa el punto terrestre más cercano (43.3, -8.4) en lugar de las coordenadas exactas
CIUDADES = {
    'A Coruña': {'lat': 43.3,  'lon': -8.4},
    'Vigo':     {'lat': 42.23, 'lon': -8.72},
    'Ourense':  {'lat': 42.34, 'lon': -7.86},
    'Santiago': {'lat': 42.88, 'lon': -8.54},
    'Lugo':     {'lat': 43.01, 'lon': -7.55},
}

ruta_temp = '/content/temp_procesado/'
os.makedirs(ruta_temp, exist_ok=True)

# Obtener lista de ZIPs ordenados por año
zips_disponibles = sorted([
    f for f in os.listdir(RUTA_BASE)
    if f.startswith('era5_diario_') and f.endswith('.zip')
])

print(f"Ficheros encontrados: {len(zips_disponibles)}")
print(f"Rango: {zips_disponibles[0]} → {zips_disponibles[-1]}")

todas_las_series = []

for zip_fichero in zips_disponibles:
    año = zip_fichero.replace('era5_diario_', '').replace('.zip', '')
    ruta_zip = RUTA_BASE + zip_fichero

    try:
        # Descomprimir el ZIP en carpeta temporal
        with zipfile.ZipFile(ruta_zip, 'r') as z:
            nombre_nc = z.namelist()[0]
            z.extractall(ruta_temp)

        # Abrir el fichero NetCDF
        ds = xr.open_dataset(ruta_temp + nombre_nc)

        # Extraer las series de las 5 ciudades convirtiendo de Kelvin a Celsius
        series_año = {}
        for ciudad, coords in CIUDADES.items():
            punto = ds['t2m'].sel(
                latitude=coords['lat'],
                longitude=coords['lon'],
                method='nearest'
            )
            serie = punto.to_series() - 273.15
            serie.index = pd.to_datetime(serie.index)
            series_año[ciudad] = serie

        df_año = pd.DataFrame(series_año)
        todas_las_series.append(df_año)

        # Cerrar y eliminar el fichero temporal
        ds.close()
        os.remove(ruta_temp + nombre_nc)

        print(f"{año} — {len(df_año)} días procesados")

    except Exception as e:
        print(f"{año} — ERROR: {e}")

# Unir todos los años en un único DataFrame ordenado por fecha
df_completo = pd.concat(todas_las_series).sort_index()
df_completo.index.name = 'fecha'

print(f"\nDataFrame completo:")
print(f"  Filas totales: {len(df_completo)} días")
print(f"  Periodo: {df_completo.index[0].date()} → {df_completo.index[-1].date()}")
print(f"  Valores nulos: {df_completo.isna().sum().sum()}")

In [ ]:
# Guardar el DataFrame unificado como CSV
ruta_csv = RUTA_BASE + 'datos/temperaturas_verano_completo.csv'
os.makedirs(RUTA_BASE + 'datos/', exist_ok=True)
df_completo.to_csv(ruta_csv)

print(f"CSV guardado en: {ruta_csv}")
print(f"Tamaño: {os.path.getsize(ruta_csv) / 1024 / 1024:.1f} MB")
print("\nPrimeras filas:")
print(df_completo.head(10).round(1))

## 4. Verificación final

Comprobación rápida de que el CSV se generó correctamente.

In [ ]:
# Recargar el CSV y verificar
df_check = pd.read_csv(ruta_csv, index_col='fecha', parse_dates=True)

print("Verificación del CSV generado:")
print(f"  Filas: {len(df_check)}")
print(f"  Columnas: {list(df_check.columns)}")
print(f"  Periodo: {df_check.index[0].date()} → {df_check.index[-1].date()}")
print(f"  Valores nulos: {df_check.isna().sum().sum()}")
print(f"\nRangos de temperatura por ciudad (°C):")
print(df_check.describe().round(1))
print("\nFichero listo para usar en 02_noches_tropicales_analisis.ipynb")